In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# エラー軽減の設定

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** 新しい実行モデルのベータリリースが利用可能になりました。指向性実行モデルは、エラー軽減ワークフローのカスタマイズにおいてより高い柔軟性を提供します。詳細については[指向性実行モデル](/guides/directed-execution-model)ガイドを参照してください。


<details>
<summary><b>パッケージバージョン</b></summary>

このページのコードは以下の要件を使用して開発されました。
これらのバージョンまたはそれ以降の使用を推奨します。

```
qiskit-ibm-runtime~=0.43.1
```
</details>
エラー軽減技術により、ユーザーは実行時のデバイスノイズをモデリングして
回路エラーを軽減できます。これは通常、モデルトレーニングに関連する量子
前処理オーバーヘッドと、生成されたモデルを使用して生の結果のエラーを
軽減するための古典的な後処理オーバーヘッドをもたらします。

Estimatorプリミティブは、[TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)、[ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)、[PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)、[PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)を含む複数のエラー軽減技術をサポートしています。各手法の説明については
[エラー軽減およびエラー抑制技術](error-mitigation-and-suppression-techniques)を
参照してください。プリミティブを使用する場合、個々の手法をオンまたはオフに
できます。詳細は[カスタムエラー設定](#advanced-error)セクションを参照してください。

> **Note:** Samplerはエラー軽減をサポートしていませんが、[mthree](https://qiskit.github.io/qiskit-addon-mthree/)（行列フリー測定軽減）パッケージを使用してローカルでエラー軽減を実行できます。

Estimatorは`resilience_level`もサポートしています。レジリエンスレベルは、
エラーに対してどの程度の耐性を構築するかを指定します。より高いレベルはより
正確な結果を生成しますが、処理時間が長くなります。レジリエンスレベルは、
プリミティブクエリにエラー軽減を適用する際のコスト/精度のトレードオフを
設定するために使用できます。エラー軽減は、関連する回路のコレクション
（アンサンブル）からの出力を処理することで、結果のエラー（バイアス）を
削減します。エラー削減の程度は適用される手法によって異なります。レジリエンス
レベルは、エラー軽減手法の詳細な選択を抽象化し、ユーザーがアプリケーションに
適切なコスト/精度のトレードオフについて判断できるようにします。

これを踏まえ、各レベルは量子サンプリングオーバーヘッドが増加する手法に
対応しており、異なる時間精度のトレードオフを試すことができます。以下の表は、
各プリミティブで利用可能なレベルと対応する手法を示しています。

> **Info:** エラー軽減はタスク固有であるため、適用できる技術は分布のサンプリングか
> 期待値の生成かによって異なります。

<span id="resilience-table"></span>

Estimatorは以下のレジリエンスレベルをサポートしています。Samplerは
レジリエンスレベルをサポートしていません。

| レジリエンスレベル | 定義                                                                                            | 技術                                                                 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | 軽減なし                                                                                         | なし                                                                      |
| 1 [デフォルト]      | 最小限の軽減コスト：読み取りエラーに関連するエラーを軽減                               | Twirled Readout Error eXtinction（TREX）測定トワーリング             |
| 2                | 中程度の軽減コスト。通常、推定量のバイアスを削減しますが、ゼロバイアスは保証されません。 | レベル1 + Zero Noise Extrapolation（ZNE）およびゲートトワーリング

> **Info:** レジリエンスレベルは現在ベータ版であるため、サンプリングオーバーヘッドとソリューション品質は回路ごとに異なります。新機能、高度なオプション、および管理ツールは順次リリースされます。各レジリエンスレベルで特定のエラー軽減手法が適用されることは保証されていません。

## レジリエンスレベルによるEstimatorの設定
レジリエンスレベルを使用してエラー軽減技術を指定するか、[カスタムエラー設定](#advanced-error)で説明されているように個別のカスタム技術を設定できます。

<details>
<summary>レジリエンスレベル0</summary>

ユーザープログラムにエラー軽減は適用されません。

</details>

<details>
<summary>レジリエンスレベル1</summary>

レベル1は、Twirled Readout Error eXtinction（TREX）として知られるモデルフリー
技術を適用して、**読み取りエラー軽減**と**測定トワーリング**を行います。測定
直前にXゲートを通じてランダムに量子ビットを反転させることで、測定に関連する
ノイズチャネルを対角化し、測定エラーを削減します。対角ノイズチャネルからの
再スケーリング項は、ゼロ状態で初期化されたランダム回路のベンチマーキングに
よって学習されます。これにより、サービスは読み取りノイズに起因する期待値の
バイアスを除去できます。このアプローチの詳細は[Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738)で説明されています。

</details>

<details>
<summary>レジリエンスレベル2</summary>

レベル2は、**レベル1に含まれるエラー軽減技術**に加えて、**ゲートトワーリング**
と**Zero Noise Extrapolation（ZNE）手法**を適用します。ZNEは、異なるノイズ
ファクター（増幅段階）でオブザーバブルの期待値を計算し、測定された期待値を
使用してゼロノイズ限界での理想的な期待値を推定します（外挿段階）。この
アプローチは期待値のエラーを削減する傾向がありますが、不偏な結果を生成する
ことは保証されていません。

![この画像はグラフを示しています。x軸はノイズ増幅ファクター、y軸は期待値です。上昇する線は軽減された値とラベル付けされています。線の近くの点はノイズ増幅された値です。x軸のすぐ上に正確な値とラベル付けされた水平線があります。](../docs/images/guides/configure-error-mitigation/resilience-2.svg "ZNE手法の図解")

この手法のオーバーヘッドはノイズファクターの数に応じてスケールします。デフォルト設定では3つのノイズファクターで期待値をサンプリングするため、このレジリエンスレベルを使用する場合、おおよそ3倍のオーバーヘッドが発生します。

レベル2では、TREX手法は測定直前にXゲートを通じてランダムに量子ビットを反転させ、Xゲートが適用された場合は対応する測定ビットを反転させます。このアプローチの詳細は[Model-free readout-error mitigation for quantum expectation values](https://arxiv.org/abs/2012.09738)で説明されています。

</details>

### 例
`EstimatorV2`インターフェースにより、ユーザーはオブザーバブルの期待値のエラーを削減するために、さまざまなエラー軽減手法をシームレスに利用できます。以下のコードは、`resilience_level 2`を設定するだけでZero Noise Extrapolationと読み取りエラー軽減を使用します。

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## カスタムエラー設定
動的デカップリング、ゲートおよび測定トワーリング、測定エラー軽減、PEC、ZNEを
含む個々のエラー軽減およびエラー抑制手法をオンまたはオフにできます。各手法の
説明については[エラー軽減およびエラー抑制技術](error-mitigation-and-suppression-techniques)を参照してください。

> **Note:** - すべてのオプションが両方のプリミティブで利用できるわけではありません。利用可能なオプションのリストについては、[利用可能なオプションの表](runtime-options-overview#options-table)を参照してください。
> - すべての手法がすべてのタイプの回路で連携して動作するわけではありません。詳細は[機能互換性テーブル](runtime-options-overview#options-compatibility-table)を参照してください。

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">